# Notebook 1: Quantum Cryptographic Risk Analysis

This notebook demonstrates the `quantum_crypto` module, exploring post-quantum
cryptographic vulnerability timelines, Harvest-Now-Decrypt-Later (HNDL) attack
windows, and migration scheduling optimization.

**Target audience:** Quantitative researchers, risk managers, and security practitioners
evaluating PQC migration urgency for financial institution portfolios.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from core.scenario import QuantumCryptoParams
from quantum_crypto.pqc_vulnerability import PQCVulnerabilityModel, QUANTUM_SECURITY_HALF_LIVES
from quantum_crypto.harvest_now_decrypt import HNDLModel
from quantum_crypto.migration_scheduler import MigrationScheduler

rng = np.random.default_rng(42)
print('Modules loaded successfully.')

## 1. Encryption Standard Vulnerability Comparison

Compare quantum security half-lives across common encryption standards.

In [ ]:
standards = list(QUANTUM_SECURITY_HALF_LIVES.keys())
half_lives = list(QUANTUM_SECURITY_HALF_LIVES.values())

fig = px.bar(
    x=standards, y=half_lives,
    title='Quantum Security Half-Lives by Encryption Standard',
    labels={'x': 'Encryption Standard', 'y': 'Security Half-Life (years)'},
    color=half_lives,
    color_continuous_scale='RdYlGn'
)
fig.add_hline(y=7, line_dash='dash', annotation_text='Median CRQC estimate (7yr)')
fig.show()

## 2. Breach Probability Over Time — Scenario Comparison

In [ ]:
scenarios = {
    'RSA-2048 (no migration)': QuantumCryptoParams(encryption_standard='RSA-2048', migration_budget_usd=0),
    'RSA-2048 (active migration)': QuantumCryptoParams(encryption_standard='RSA-2048', migration_budget_usd=5_000_000),
    'AES-256 (well-positioned)': QuantumCryptoParams(encryption_standard='AES-256', migration_budget_usd=2_000_000),
    'ECC-256 (high exposure)': QuantumCryptoParams(encryption_standard='ECC-256', migration_budget_usd=500_000),
}

fig = go.Figure()
for name, params in scenarios.items():
    model = PQCVulnerabilityModel(params, rng=np.random.default_rng(42))
    probs = model.compute_breach_probability_by_year()
    years = list(range(1, len(probs) + 1))
    fig.add_trace(go.Scatter(x=years, y=probs, name=name, mode='lines+markers'))

fig.update_layout(
    title='Annual Breach Probability by Encryption Standard and Migration Status',
    xaxis_title='Year from Present',
    yaxis_title='Breach Probability',
    yaxis=dict(tickformat='.0%')
)
fig.show()

## 3. HNDL Attack Window Analysis

In [ ]:
# Sweep migration completion time vs. HNDL exposure
migration_years_range = np.linspace(0.5, 8, 20)
hndl_urgency_scores = []

for myr in migration_years_range:
    model = HNDLModel(
        data_shelf_life_years=20,
        years_to_crqc=7,
        migration_years_remaining=myr,
        rng=np.random.default_rng(42)
    )
    exposure = model.compute_exposure()
    hndl_urgency_scores.append(exposure.urgency_score)

fig = px.line(
    x=migration_years_range, y=hndl_urgency_scores,
    title='HNDL Urgency Score vs. Migration Timeline',
    labels={'x': 'Years Remaining to Complete PQC Migration', 'y': 'HNDL Urgency Score'}
)
fig.add_vline(x=3, line_dash='dash', annotation_text='Typical migration timeline')
fig.show()

## 4. PQC Migration Plan — Asset Prioritization

In [ ]:
scheduler = MigrationScheduler(
    budget_usd=2_000_000,
    target_algorithm='CRYSTALS-Kyber',
    years_available=5,
    rng=np.random.default_rng(42)
)
plan = scheduler.compute_plan()

print(f'Total migration cost: ${plan.total_cost_usd:,.0f}')
print(f'Migration timeline: {plan.timeline_years} years')
print('\nRecommended migration order:')
for i, asset in enumerate(plan.recommended_order, 1):
    print(f'  {i}. {asset}')

fig = px.line(
    x=list(range(1, len(plan.risk_reduction_curve) + 1)),
    y=plan.risk_reduction_curve,
    title='Projected Risk Reduction Curve Over Migration Timeline',
    labels={'x': 'Year', 'y': 'Cumulative Risk Reduction'}
)
fig.update_layout(yaxis=dict(tickformat='.0%'))
fig.show()